# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## Data Description

The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.
- **Store_Type** - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **Installing and Importing the necessary libraries**

In [ ]:
# Install only packages that may not already be available in Google Colab.
# Core packages (NumPy, pandas, SciPy, scikit-learn, matplotlib, seaborn)
# are intentionally left at the Colab runtime's compatible versions.
!pip install -q xgboost joblib requests flask streamlit huggingface_hub


**Note:**

- This completed version uses the compatible scientific-Python stack supplied by the current Google Colab runtime.
- Only additional packages required by the project are installed above. This avoids forcing older NumPy/SciPy/scikit-learn versions into a newer Colab Python runtime.
- Run the notebook sequentially from top to bottom so that all variables, transformations, models, and deployment artifacts are created in the correct order.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset and tuning models
from sklearn.model_selection import train_test_split, GridSearchCV

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Regression models
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

# Regression metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

# Preprocessing and pipelines
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Model serialization
import joblib

# General utilities
import os
import json
import requests


# **Loading the dataset**

In [ ]:
# Load the SuperKart training dataset
data = pd.read_csv("SuperKart.csv")

# Preserve the original data and work on a copy
df = data.copy()

print("Dataset loaded successfully.")
display(df.head())


# **Data Overview**
**Observation:** The dataset contains 8,763 rows and 12 columns. The target variable is `Product_Store_Sales_Total`, making this a supervised regression problem. The supplied data contains no missing values or duplicate rows.


In [ ]:
print("Shape of the dataset:", df.shape)
print("\nData types and non-null counts:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nNumber of duplicate rows:", df.duplicated().sum())

print("\nUnique values in categorical columns:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(20))


# **Exploratory Data Analysis (EDA)**

## Univariate Analysis

In [ ]:
# Separate numerical and categorical variables
numerical_cols = [
    "Product_Weight",
    "Product_Allocated_Area",
    "Product_MRP",
    "Store_Establishment_Year",
    "Product_Store_Sales_Total",
]

categorical_cols = [
    "Product_Sugar_Content",
    "Product_Type",
    "Store_Size",
    "Store_Location_City_Type",
    "Store_Type",
]

# Numerical distributions and boxplots
for col in numerical_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=df, x=col, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    sns.boxplot(data=df, x=col, ax=axes[1])
    axes[1].set_title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

# Categorical distributions
for col in categorical_cols:
    plt.figure(figsize=(10, 4))
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

print("""
Key observations:
- Product Weight and Product MRP are broadly centered around their middle ranges.
- Product Allocated Area is positively skewed, with a smaller number of products receiving much larger display allocations.
- Store Establishment Year occurs at a limited number of discrete years and is better treated as a business/time feature than as a conventional continuous measurement.
- Product Store Sales Total is concentrated around its middle range with observations at both lower and upper extremes.
- Boxplot-identified extreme values are not removed automatically because they may represent legitimate retail observations.
- Product_Sugar_Content contains an inconsistent 'reg' label that should be standardized to 'Regular' during preprocessing.
""")


## Bivariate Analysis

In [ ]:
target = "Product_Store_Sales_Total"

# Numerical features versus target
for col in ["Product_Weight", "Product_Allocated_Area", "Product_MRP", "Store_Establishment_Year"]:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df, x=col, y=target, alpha=0.5)
    plt.title(f"{col} vs {target}")
    plt.tight_layout()
    plt.show()

# Correlation among numerical variables
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# Categorical features versus target
for col in categorical_cols:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=col, y=target)
    plt.title(f"{col} vs {target}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

print("""
Key observations:
- Product MRP shows a strong positive association with Product Store Sales Total and is expected to be an important predictor.
- Product Weight has a positive but more dispersed relationship with sales.
- Product Allocated Area does not show a strong simple linear relationship with sales when considered independently.
- Store Establishment Year forms discrete groups; converting it to store age provides a more interpretable representation.
- Store-related categorical variables, particularly Store Type, Store Size, and City Tier, show meaningful differences in sales distributions.
- Product Type also contributes information, although its category distributions overlap substantially.
- The 'reg' value in Product Sugar Content is treated as an inconsistent representation of 'Regular' and will be standardized.
""")


# **Data Preprocessing**
**Preprocessing approach:** Standardize the inconsistent `reg` sugar-content label, engineer interpretable product/store features, remove raw identifiers replaced by engineered features, split the data before model fitting, and place one-hot encoding inside the modeling pipeline to ensure consistent transformations.


In [ ]:
# -----------------------------
# 1. Clean categorical labels
# -----------------------------
df["Product_Sugar_Content"] = df["Product_Sugar_Content"].replace({"reg": "Regular"})

# -----------------------------
# 2. Feature engineering
# -----------------------------

# Product IDs begin with two letters. Preserve the informative prefix and remove
# the unique identifier itself.
df["Product_Id_char"] = df["Product_Id"].str[:2]

# Convert establishment year to store age.
# 2025 is used as the project reference year.
df["Store_Age_Years"] = 2025 - df["Store_Establishment_Year"]

# Consolidate detailed product types into broader perishability categories.
perishable_products = [
    "Dairy",
    "Meat",
    "Fruits and Vegetables",
    "Breakfast",
    "Seafood",
    "Frozen Foods",
    "Breads",
]
df["Product_Type_Category"] = np.where(
    df["Product_Type"].isin(perishable_products),
    "Perishables",
    "Non Perishables",
)

# Remove raw identifiers and columns replaced by engineered features
df.drop(
    columns=["Product_Id", "Store_Id", "Store_Establishment_Year", "Product_Type"],
    inplace=True,
)

print("Cleaned sugar-content categories:")
print(df["Product_Sugar_Content"].value_counts())

print("\nFinal modeling columns:")
print(df.columns.tolist())

# -----------------------------
# 3. Define predictors and target
# -----------------------------
X = df.drop(columns=["Product_Store_Sales_Total"])
y = df["Product_Store_Sales_Total"]

# -----------------------------
# 4. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = X_train.select_dtypes(include="object").columns.tolist()
numerical_features = X_train.select_dtypes(exclude="object").columns.tolist()

# One-hot encode categorical features while passing numerical features through.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough",
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)


# **Model Building**
**Modeling approach:** Random Forest and Gradient Boosting regressors are built and compared using RMSE, MAE, R-squared, adjusted R-squared, and MAPE. The same preprocessing pipeline is applied to both models.


## Define functions for Model Evaluation

In [ ]:
# Function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))

# Function to compute regression performance metrics
def model_performance_regression(model, predictors, target):
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)
    adjr2 = adj_r2_score(predictors, target, pred)
    rmse = np.sqrt(mean_squared_error(target, pred))
    mae = mean_absolute_error(target, pred)
    mape = mean_absolute_percentage_error(target, pred)

    return pd.DataFrame(
        {
            "RMSE": [rmse],
            "MAE": [mae],
            "R-squared": [r2],
            "Adj. R-squared": [adjr2],
            "MAPE": [mape],
        }
    )


The ML models to be built can be any two out of the following:
1. Decision Tree
2. Bagging
3. Random Forest
4. AdaBoost
5. Gradient Boosting
6. XGBoost

In [ ]:
# Build two ensemble regression models using the same preprocessing pipeline

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )),
    ]
)

gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(random_state=42)),
    ]
)

# Fit models
rf_pipeline.fit(X_train, y_train)
gb_pipeline.fit(X_train, y_train)

# Evaluate on training and test data
rf_train_perf = model_performance_regression(rf_pipeline, X_train, y_train)
rf_test_perf = model_performance_regression(rf_pipeline, X_test, y_test)

gb_train_perf = model_performance_regression(gb_pipeline, X_train, y_train)
gb_test_perf = model_performance_regression(gb_pipeline, X_test, y_test)

comparison_before_tuning = pd.concat(
    [
        rf_train_perf.assign(Model="Random Forest - Train"),
        rf_test_perf.assign(Model="Random Forest - Test"),
        gb_train_perf.assign(Model="Gradient Boosting - Train"),
        gb_test_perf.assign(Model="Gradient Boosting - Test"),
    ],
    ignore_index=True,
).set_index("Model")

display(comparison_before_tuning)

print("""
Model-building interpretation:
The models are compared primarily using RMSE, MAE and R-squared on unseen test data.
A useful model should achieve low prediction errors and a high R-squared while also
maintaining a reasonable gap between training and test performance.
""")


# **Model Performance Improvement - Hyperparameter Tuning**
**Tuning approach:** Hyperparameters are selected using cross-validated RMSE on the training data. The held-out test set is reserved for final performance comparison.


In [ ]:
# Hyperparameter tuning of Random Forest
# Random Forest is tuned because its baseline performance is strong and it can
# model nonlinear interactions among product and store characteristics.

rf_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

param_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [12, 18, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", 0.8],
}

grid_search = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("\nBest cross-validation RMSE:", -grid_search.best_score_)

tuned_rf = grid_search.best_estimator_

tuned_rf_train_perf = model_performance_regression(tuned_rf, X_train, y_train)
tuned_rf_test_perf = model_performance_regression(tuned_rf, X_test, y_test)

display(
    pd.concat(
        [
            tuned_rf_train_perf.assign(Model="Tuned Random Forest - Train"),
            tuned_rf_test_perf.assign(Model="Tuned Random Forest - Test"),
        ],
        ignore_index=True,
    ).set_index("Model")
)


# **Model Performance Comparison, Final Model Selection, and Serialization**
**Selection principle:** Compare test-set errors and explanatory performance after tuning, then serialize the complete preprocessing-and-model pipeline so deployment receives exactly the same transformations used during training.


In [ ]:
# Compare baseline and tuned models on the test set
model_comparison = pd.concat(
    [
        rf_test_perf.assign(Model="Random Forest"),
        gb_test_perf.assign(Model="Gradient Boosting"),
        tuned_rf_test_perf.assign(Model="Tuned Random Forest"),
    ],
    ignore_index=True,
).set_index("Model")

display(model_comparison.sort_values("RMSE"))

# Select the tuned Random Forest as the final candidate after CV tuning.
# Its exact performance should be verified from the output above when the notebook is run.
final_model = tuned_rf

# Serialize the complete preprocessing + model pipeline
joblib.dump(final_model, "superkart_model.joblib")

print("Final model pipeline saved as: superkart_model.joblib")


# **Deployment - Backend**

## Points to note before executing the below cells
- Create a GitHub account if you don't already have one at [github.com](https://github.com)
- Generate a **Personal Access Token**
  - Make sure the token has **repo** scope (full control of private repositories)
- The serialized ML model file (`superkart_model.joblib`) should already be present in the `backend_files` folder before pushing to GitHub
- We will deploy **both the Flask backend and the Streamlit frontend as separate Docker containers** inside a GitHub Codespace, and connect them using a **Docker network**

## Flask Web Framework

Flask is a lightweight, flexible Python web framework used to build web applications and APIs quickly and easily.

Flask allows you to:
- Create web routes (URLs that users can access)
- Handle HTTP requests and responses
- Build REST APIs to expose machine learning models or other services

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
import pandas as pd
import joblib
import io

superkart_api = Flask(__name__)
model = joblib.load("superkart_model.joblib")

FEATURES = [
    "Product_Weight",
    "Product_Sugar_Content",
    "Product_Allocated_Area",
    "Product_MRP",
    "Store_Size",
    "Store_Location_City_Type",
    "Store_Type",
    "Product_Id_char",
    "Store_Age_Years",
    "Product_Type_Category",
]

@superkart_api.get("/")
def health():
    return jsonify({"status": "SuperKart Sales Prediction API is running"})

@superkart_api.post("/v1/predict")
def predict():
    try:
        payload = request.get_json(force=True)
        input_df = pd.DataFrame([payload])[FEATURES]
        prediction = float(model.predict(input_df)[0])
        return jsonify({"predicted_sales": round(prediction, 2)})
    except Exception as exc:
        return jsonify({"error": str(exc)}), 400

@superkart_api.post("/v1/predictbatch")
def predict_batch():
    try:
        uploaded_file = request.files["file"]
        batch_df = pd.read_csv(io.BytesIO(uploaded_file.read()))
        predictions = model.predict(batch_df[FEATURES])
        result = {str(i): round(float(v), 2) for i, v in enumerate(predictions)}
        return jsonify(result)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 400

if __name__ == "__main__":
    superkart_api.run(host="0.0.0.0", port=7860)


## Dependencies File

In [ ]:
%%writefile requirements.txt
flask
pandas
numpy
scikit-learn
joblib
gunicorn


## Dockerfile

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY superkart_model.joblib .

EXPOSE 7860

CMD ["gunicorn", "--bind", "0.0.0.0:7860", "app:superkart_api"]


# **Deployment - Frontend**

## Streamlit for Interactive UI

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import requests

st.set_page_config(page_title="SuperKart Sales Predictor", page_icon="🛒")
st.title("SuperKart Product-Store Sales Predictor")
st.write("Enter product and store characteristics to estimate sales revenue.")

api_url = st.text_input(
    "Backend prediction endpoint",
    value="http://localhost:7860/v1/predict"
)

product_weight = st.number_input("Product Weight", min_value=0.0, value=12.66)
sugar = st.selectbox("Product Sugar Content", ["Low Sugar", "Regular", "No Sugar"])
allocated_area = st.number_input("Product Allocated Area", min_value=0.0, value=0.027, format="%.3f")
mrp = st.number_input("Product MRP", min_value=0.0, value=117.08)
store_size = st.selectbox("Store Size", ["Small", "Medium", "High"])
city_type = st.selectbox("Store Location City Type", ["Tier 1", "Tier 2", "Tier 3"])
store_type = st.selectbox(
    "Store Type",
    ["Food Mart", "Supermarket Type1", "Supermarket Type2", "Departmental Store"]
)
product_id_char = st.selectbox("Product ID Prefix", ["FD", "DR", "NC"])
store_age = st.number_input("Store Age (Years)", min_value=0, value=16)
product_category = st.selectbox(
    "Product Type Category", ["Perishables", "Non Perishables"]
)

if st.button("Predict Sales"):
    payload = {
        "Product_Weight": product_weight,
        "Product_Sugar_Content": sugar,
        "Product_Allocated_Area": allocated_area,
        "Product_MRP": mrp,
        "Store_Size": store_size,
        "Store_Location_City_Type": city_type,
        "Store_Type": store_type,
        "Product_Id_char": product_id_char,
        "Store_Age_Years": store_age,
        "Product_Type_Category": product_category,
    }
    try:
        response = requests.post(api_url, json=payload, timeout=30)
        response.raise_for_status()
        prediction = response.json()["predicted_sales"]
        st.success(f"Predicted Sales Revenue: {prediction:,.2f}")
    except Exception as exc:
        st.error(f"Prediction request failed: {exc}")


## Dependencies File

In [ ]:
%%writefile requirements_frontend.txt
streamlit
requests


## Dockerfile

In [ ]:
%%writefile Dockerfile.frontend
FROM python:3.11-slim

WORKDIR /app

COPY requirements_frontend.txt .
RUN pip install --no-cache-dir -r requirements_frontend.txt

COPY streamlit_app.py .

EXPOSE 8501

CMD ["streamlit", "run", "streamlit_app.py", "--server.address=0.0.0.0", "--server.port=8501"]


# **Pushing Deployment Files to GitHub**

In [ ]:
# Prepare deployment artifacts for GitHub/Codespaces.
# Authentication credentials are intentionally NOT hard-coded in the notebook.

deployment_files = [
    "superkart_model.joblib",
    "app.py",
    "requirements.txt",
    "Dockerfile",
    "streamlit_app.py",
    "requirements_frontend.txt",
    "Dockerfile.frontend",
]

print("Deployment files created:")
for file_name in deployment_files:
    print(f"{file_name}: {'FOUND' if os.path.exists(file_name) else 'MISSING'}")

print("""
Next deployment step:
1. Create/open your GitHub repository or Codespace.
2. Upload the deployment files listed above.
3. Build/run the backend container on port 7860.
4. Build/run the frontend container on port 8501.
5. Forward the required ports and paste the forwarded backend URL into the
   model_root_url cell below.
""")


# **Inferencing using Flask API**

As the ***frontend and backend are decoupled***, we can ***access the backend directly for predictions***.
- The decoupling ensures seamless interaction with the deployed model while leveraging the API for scalable inference.

Let's see how to interact with the deployed Flask API running inside a GitHub Codespace to perform **online** and **batch inference** from this Colab notebook.

We will:
1. Send API requests for both online and batch inference.
2. Process and check the model predictions.

**Before running the cells below**, make sure:
- Your Codespace is running with both containers up
- Port 7860 has been set to **Public** in the Codespace Ports tab
- You have copied the forwarded URL for port 7860
- You have also referred the document titled **`Guided Hands-on - Containerized Model Deployment using GitHub Codespaces`** provided

In [ ]:
import json  # To handle JSON formatting for API requests and responses
import requests  # To send HTTP requests to the deployed Flask API

import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical computations

In [ ]:
# Replace the value below with the forwarded URL for backend port 7860
# after the Flask container is running in your GitHub Codespace.
model_root_url = "PASTE_YOUR_CODESPACE_7860_URL_HERE"


Replace the URL above with the actual forwarded URL from your Codespace's Ports tab. It will look something like `https://organic-space-abcd1234-7860.app.github.dev`. Make sure there is no trailing slash at the end.

This URL is a tunnel that Codespaces creates to route external HTTP requests into your Codespace and to the Flask container running on port 7860.

In [ ]:
model_url = model_root_url + "/v1/predict"  # Endpoint for online (single) inference

Since our model predictions are provided by the following endpoint we created using Flask, we need to invoke the same to make prediction.

> ```@superkart_api.post('/v1/predict')```

In [ ]:
model_batch_url = model_root_url + "/v1/predictbatch"  # Endpoint for batch inference

> ```@superkart_api.post('/v1/predictbatch')```

## Online

**Online Inference:** Sending a single request to the API and receiving an immediate response. This is useful for real-time applications like recommendation systems and fraud detection.  
* This data is sent as a JSON payload in a POST request to the model endpoint.

* The model processes the input features and returns a prediction.

In [ ]:
payload = {
  "Product_Weight": 12.66,
  "Product_Sugar_Content": "Low Sugar",
  "Product_Allocated_Area": 0.027,
  "Product_MRP": 117.08,
  "Store_Size": "Small",
  "Store_Location_City_Type": "Tier 1",
  "Store_Type": "Supermarket Type1",
  "Product_Id_char": "FD",
  "Store_Age_Years": 16,
  "Product_Type_Category": "Perishables"
}


In [ ]:
# Sending a POST request to the model endpoint with the test payload
response = requests.post(model_url, json=payload)

In [ ]:
response

`<Response [200]>` indicates that the HTTP request was successful.  

- `Response` is the object returned by `requests.post()`.  
- `[200]` is the HTTP status code, meaning **OK** (the request was processed successfully).  

In [ ]:
print(response.json())

`response.json()` is used to parse the response body as JSON.  
- If the API returns data in JSON format, `.json()` converts it into a Python dictionary.  
- This allows easy access to specific values using keys.  

## Batch

**Batch Inference:** Sending multiple inputs in a single request, allowing efficient processing of large datasets. This is ideal for analyzing historical data at scale.

In [ ]:
batch_dataset = pd.read_csv("Batch_Data_SuperKart.csv")

**Note**: The batch CSV file must contain all the feature columns expected by the model: `Product_Weight`, `Product_Sugar_Content`, `Product_Allocated_Area`, `Product_MRP`, `Store_Size`, `Store_Location_City_Type`, `Store_Type`, `Product_Id_char`, `Store_Age_Years`, `Product_Type_Category`.

In [ ]:
# Prepare batch input for API request
batch_input = {
    'file': batch_dataset.to_csv(header=True, index=False).encode('utf-8')
}

This code prepares the product data as a **file-like object** to send in an API request (in the `files` parameter of `requests.post`).

- `batch_dataset`: This is a Pandas DataFrame containing the product data loaded from a CSV file.

- `.to_csv(header=True, index=False)`: Converts the DataFrame into a CSV **string**, keeping the column headers (`header=True`) but **excluding the index** (`index=False`).

- `.encode('utf-8')`: Encodes the CSV string into **bytes** (UTF-8 format), which is necessary for sending it over an HTTP request.

- `'file': ...`: Wraps the encoded CSV data in a dictionary with the key `'file'`, which is expected by the Flask backend.

In [ ]:
# Send request to the model API for batch predictions
response = requests.post(
    model_batch_url,  # Model endpoint URL
    files=batch_input
)

This line sends a **POST request** to the deployed model API to perform **batch sales prediction**.

- `requests.post(...)`: This uses the `requests` library to make a **POST** request to a web server (in this case, our API endpoint for batch predictions).

- `model_batch_url`: This is the URL endpoint where the batch prediction API is hosted. It should look like this:\
  `"https://<your-codespace-name>-7860.app.github.dev/v1/predictbatch"`

- `files=batch_input`: This sends the batch data file (e.g., a CSV) as part of the request using the `files` parameter.

- `response`: This is the result of the API call and contains the response from the server, including predicted sales values.

In [ ]:
response

In [ ]:
# Extract predictions from API response
response.text

As we can see, we receive a JSON where each key represents a row index, and the value represents the model's predicted sales for that product.

# **Actionable Insights and Business Recommendations**

- **Pricing is a major sales driver:** Product MRP shows a strong positive relationship with product-store sales. SuperKart should incorporate price-band and product-value information when planning inventory and revenue targets, while avoiding the assumption that higher prices alone cause higher sales.

- **Store characteristics matter materially:** Store type, store size, and city tier show meaningful differences in sales distributions. Inventory allocation and regional sales planning should therefore be differentiated by store format and location rather than using one chain-wide strategy.

- **Use forecasts to improve replenishment decisions:** The final model can provide product-store level revenue estimates for the upcoming quarter. These estimates can support inventory prioritization, replenishment planning, and identification of combinations where expected demand is relatively high or low.

- **Preserve product segmentation in operational use:** Product ID prefixes and broader perishability categories provide useful product-group information without retaining arbitrary unique product identifiers. The same transformations must be used consistently for both online and batch predictions.

- **Monitor prediction error after deployment:** Actual sales should be compared with predicted sales on a recurring basis. Material changes in RMSE, MAE, R-squared, or the distribution of important input variables may indicate data drift or a need to retrain the model.

- **Treat the model as decision support:** Forecasts should be combined with current promotions, supply constraints, seasonal effects, and local management knowledge that are not represented in the historical dataset.
